In [12]:

from pathlib import Path
import pandas as pd

current = Path.cwd()

ROOT = next(
    p for p in [current, *current.parents]
    if p.name.lower() == "retailflow"
)

SILVER_PATH = ROOT / "data" / "silver"
GOLD_PATH = ROOT / "data" / "gold"

GOLD_PATH.mkdir(parents=True, exist_ok=True)

# Load all Silver datasets
data = {}

for file in SILVER_PATH.glob("*.parquet"):
    name = file.stem.replace("_dataset", "")
    data[name] = pd.read_parquet(file)

print("Silver datasets loaded:", len(data))
print("Available datasets:")

for name, df in data.items():
    print(name, ":", df.shape)


Silver datasets loaded: 8
Available datasets:
olist_customers : (99441, 5)
olist_geolocation : (738332, 5)
olist_orders : (99441, 8)
olist_order_payments : (103886, 5)
olist_order_reviews : (99224, 7)
olist_products : (32951, 9)
olist_sellers : (3095, 4)
product_category_name_translation : (71, 2)


In [13]:

orders = data["olist_orders"]
customers = data["olist_customers"]
payments = data["olist_order_payments"]
items = data.get("olist_order_items")
products = data["olist_products"]
categories = data["product_category_name_translation"]

# Prepare order-level item information when that source is available
if items is not None:
    item_summary = items.groupby("order_id").agg(
        total_items=("order_item_id", "count"),
        product_revenue=("price", "sum"),
        shipping_cost=("freight_value", "sum")
    ).reset_index()

# Prepare payment information
payment_summary = payments.groupby("order_id").agg(
    payment_total=("payment_value", "sum"),
    payment_type=("payment_type", "first")
).reset_index()

# Prepare order summary
if items is not None:
    gold_orders = orders.merge(
        item_summary, on="order_id", how="left"
    )
else:
    gold_orders = orders.copy()
    for column in ["total_items", "product_revenue", "shipping_cost"]:
        gold_orders[column] = pd.NA
    print("Warning: order-item metrics are unavailable; source dataset is missing.")

gold_orders = gold_orders.merge(
    payment_summary, on="order_id", how="left"
)

gold_orders = gold_orders.merge(
    customers[["customer_id", "customer_unique_id", "customer_state"]],
    on="customer_id", how="left"
)

gold_orders["order_purchase_timestamp"] = pd.to_datetime(
    gold_orders["order_purchase_timestamp"],
    errors="coerce"
)

if items is not None:
    gold_orders["total_items"] = gold_orders["total_items"].fillna(0)
    gold_orders["product_revenue"] = gold_orders["product_revenue"].fillna(0)
    gold_orders["shipping_cost"] = gold_orders["shipping_cost"].fillna(0)
gold_orders["payment_total"] = gold_orders["payment_total"].fillna(0)

print("Business data prepared!")
print("Total orders:", len(gold_orders))


Business data prepared!
Total orders: 99441


In [14]:

# Use delivered orders for completed sales analysis
delivered = gold_orders[
    gold_orders["order_status"] == "delivered"
].copy()

# 1. Order Summary
gold_order_summary = gold_orders.copy()

# 2. Monthly Sales
delivered["sales_month"] = (
    delivered["order_purchase_timestamp"].dt.to_period("M").astype(str)
)

gold_monthly_sales = delivered.groupby("sales_month").agg(
    total_orders=("order_id", "nunique"),
    total_revenue=("product_revenue", "sum"),
    total_shipping=("shipping_cost", "sum")
).reset_index()
if items is None:
    gold_monthly_sales[["total_revenue", "total_shipping"]] = pd.NA

# 3. Category Sales
if items is not None:
    category_items = items.merge(
        orders[["order_id", "order_status"]],
        on="order_id", how="inner"
    )

    category_items = category_items[
        category_items["order_status"] == "delivered"
    ]

    category_items = category_items.merge(
        products[["product_id", "product_category_name"]],
        on="product_id", how="left"
    )

    category_items = category_items.merge(
        categories,
        on="product_category_name", how="left"
    )

    gold_category_sales = category_items.groupby(
        "product_category_name_english", dropna=False
    ).agg(
        total_items=("order_item_id", "count"),
        total_revenue=("price", "sum")
    ).reset_index()
else:
    gold_category_sales = pd.DataFrame(columns=[
        "product_category_name_english", "total_items", "total_revenue"
    ])

# 4. State Sales
gold_state_sales = delivered.groupby("customer_state").agg(
    total_orders=("order_id", "nunique"),
    total_customers=("customer_unique_id", "nunique"),
    total_revenue=("product_revenue", "sum")
).reset_index()
if items is None:
    gold_state_sales["total_revenue"] = pd.NA

# 5. Payment Analysis
gold_payment_analysis = payments.merge(
    orders[["order_id", "order_status"]],
    on="order_id", how="inner"
)

gold_payment_analysis = gold_payment_analysis[
    gold_payment_analysis["order_status"] == "delivered"
]

gold_payment_analysis = gold_payment_analysis.groupby(
    "payment_type"
).agg(
    total_transactions=("order_id", "count"),
    total_payment=("payment_value", "sum")
).reset_index()

print("All 5 Gold analytical tables created!")


All 5 Gold analytical tables created!


In [15]:

gold_tables = {
    "gold_order_summary": gold_order_summary,
    "gold_monthly_sales": gold_monthly_sales,
    "gold_category_sales": gold_category_sales,
    "gold_state_sales": gold_state_sales,
    "gold_payment_analysis": gold_payment_analysis
}

for name, df in gold_tables.items():
    output_path = GOLD_PATH / f"{name}.parquet"
    df.to_parquet(output_path, index=False)

    print(f"Saved: {name}.parquet | Rows: {len(df)}")

print("\nGOLD LAYER COMPLETED!")


Saved: gold_order_summary.parquet | Rows: 99441
Saved: gold_monthly_sales.parquet | Rows: 23
Saved: gold_category_sales.parquet | Rows: 0
Saved: gold_state_sales.parquet | Rows: 27
Saved: gold_payment_analysis.parquet | Rows: 4

GOLD LAYER COMPLETED!


In [16]:

for name, df in gold_tables.items():
    print(f"\n{name}")
    display(df.head(5))

print("\nTotal Gold tables:", len(gold_tables))
print("Gold folder:", GOLD_PATH)



gold_order_summary


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_items,product_revenue,shipping_cost,payment_total,payment_type,customer_unique_id,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,<NA>,<NA>,<NA>,38.71,credit_card,7c396fd4830fd04220f754e42b4e5bff,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,<NA>,<NA>,<NA>,141.46,boleto,af07308b275d755c9edb36a90c618231,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,<NA>,<NA>,<NA>,179.12,credit_card,3a653a41f6f9fc3d2a113cf8398680e8,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,<NA>,<NA>,<NA>,72.20,credit_card,7c142cf63193a1473d2e66489a9ae977,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,<NA>,<NA>,<NA>,28.62,credit_card,72632f0f9dd73dfee390c9b22eb56dd6,SP



gold_monthly_sales


,sales_month,total_orders,total_revenue,total_shipping
0,2016-09,1,<NA>,<NA>
1,2016-10,265,<NA>,<NA>
2,2016-12,1,<NA>,<NA>
3,2017-01,750,<NA>,<NA>
4,2017-02,1653,<NA>,<NA>



gold_category_sales


,product_category_name_english,total_items,total_revenue



gold_state_sales


,customer_state,total_orders,total_customers,total_revenue
0,AC,80,76,<NA>
1,AL,397,387,<NA>
2,AM,145,140,<NA>
3,AP,67,66,<NA>
4,BA,3256,3158,<NA>



gold_payment_analysis


,payment_type,total_transactions,total_payment
0,boleto,19191,2769932.58
1,credit_card,74586,12101094.88
2,debit_card,1486,208421.12
3,voucher,5493,343013.19



Total Gold tables: 5
Gold folder: e:\SE Academic Projects\My projects\Data Engineering 01\RetailFlow\data\gold
